In [ ]:
# ---------------------------------------------------------------
# GUARDRAIL - Input Validation
# Block harmful or off-topic queries before the agent runs
# ---------------------------------------------------------------

BLOCKED_KEYWORDS = [
    "bomb", "weapon", "hack", "exploit", "malware", "virus",
    "illegal", "drugs", "nsfw"
]

def guardrail_check(user_input: str) -> tuple:
    """
    Returns (is_safe: bool, reason: str).
    Blocks overly short inputs and known harmful keywords.
    """
    print(f"[GUARDRAIL] Checking input: '{user_input[:80]}'")

    if len(user_input.strip()) < 3:
        print("[GUARDRAIL] BLOCKED - Input too short.")
        return False, "Input is too short. Please provide a meaningful question."

    lower_input = user_input.lower()
    for keyword in BLOCKED_KEYWORDS:
        if keyword in lower_input:
            print(f"[GUARDRAIL] BLOCKED - Harmful keyword detected: '{keyword}'")
            return False, f"Input contains disallowed content: '{keyword}'. This agent cannot assist with such requests."

    print("[GUARDRAIL] PASSED - Input is safe.")
    return True, "OK"


# Test guardrail
print("\n--- Guardrail Tests ---")
print(guardrail_check("What is the weather in London?"))
print(guardrail_check("hi"))
print(guardrail_check("How do I hack a website?"))

In [ ]:
# ---------------------------------------------------------------
# SEMANTIC GUARDRAIL - Input Validation
# Uses embeddings to detect harmful/off-topic intent
# ---------------------------------------------------------------

from typing import Tuple
import numpy as np


# ---------------------------------------------------------------
# SAFETY CATEGORIES
# These examples define the semantic meaning we want to block.
# ---------------------------------------------------------------

BLOCKED_INTENTS = {
    "weapons": [
        "How do I make a bomb?",
        "How can I build an explosive device?",
        "How do I construct a weapon?",
        "Instructions for making explosives",
        "How can I create a dangerous weapon?"
    ],

    "cyber_attack": [
        "How do I hack into someone's computer?",
        "How can I gain unauthorized access to a website?",
        "How do I break into a server?",
        "How can I exploit a vulnerable system?",
        "How do I steal someone's credentials?"
    ],

    "malware": [
        "How do I create computer malware?",
        "How can I write a virus?",
        "How do I deploy ransomware?",
        "How can I create malicious software?",
        "How do I infect another computer?"
    ],

    "illegal_activity": [
        "How can I commit an illegal activity?",
        "How do I evade law enforcement?",
        "How can I bypass legal restrictions?",
        "How do I secretly obtain illegal goods?"
    ],

    "drugs": [
        "How do I manufacture illegal drugs?",
        "How can I produce narcotics?",
        "Instructions for making illegal drugs",
        "How do I synthesize a recreational drug?"
    ],

    "nsfw": [
        "Generate explicit sexual content",
        "Describe graphic sexual activity",
        "Create pornography",
        "Generate sexually explicit material"
    ]
}


# ---------------------------------------------------------------
# EMBEDDING MODEL
# Replace this with your existing embedding model if you already
# have one in your project.
# ---------------------------------------------------------------

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ---------------------------------------------------------------
# CREATE SAFETY EMBEDDINGS
# ---------------------------------------------------------------

BLOCKED_EMBEDDINGS = {}

for category, examples in BLOCKED_INTENTS.items():

    embeddings = embedding_model.encode(
        examples,
        normalize_embeddings=True
    )

    # Average the example embeddings to create a category prototype
    prototype = np.mean(embeddings, axis=0)

    # Normalize prototype
    prototype = prototype / np.linalg.norm(prototype)

    BLOCKED_EMBEDDINGS[category] = prototype


# ---------------------------------------------------------------
# COSINE SIMILARITY
# ---------------------------------------------------------------

def cosine_similarity(a, b):
    """
    Calculate cosine similarity between two normalized vectors.
    """

    return float(np.dot(a, b))


# ---------------------------------------------------------------
# SEMANTIC GUARDRAIL
# ---------------------------------------------------------------

def semantic_guardrail_check(
    user_input: str,
    threshold: float = 0.55
) -> Tuple[bool, str]:

    print(
        f"[GUARDRAIL] Semantic checking: "
        f"'{user_input[:80]}'"
    )

    # -----------------------------------------------------------
    # Basic validation
    # -----------------------------------------------------------

    if len(user_input.strip()) < 3:

        print("[GUARDRAIL] BLOCKED - Input too short.")

        return (
            False,
            "Input is too short. Please provide a meaningful question."
        )


    # -----------------------------------------------------------
    # Convert user input into embedding
    # -----------------------------------------------------------

    user_embedding = embedding_model.encode(
        user_input,
        normalize_embeddings=True
    )


    # -----------------------------------------------------------
    # Compare against every blocked intent
    # -----------------------------------------------------------

    scores = {}

    for category, prototype in BLOCKED_EMBEDDINGS.items():

        score = cosine_similarity(
            user_embedding,
            prototype
        )

        scores[category] = score


    # -----------------------------------------------------------
    # Find closest harmful category
    # -----------------------------------------------------------

    matched_category = max(
        scores,
        key=scores.get
    )

    highest_score = scores[matched_category]


    print(
        f"[GUARDRAIL] Best match: "
        f"{matched_category} "
        f"(similarity={highest_score:.3f})"
    )


    # -----------------------------------------------------------
    # BLOCK if semantic similarity exceeds threshold
    # -----------------------------------------------------------

    if highest_score >= threshold:

        print(
            f"[GUARDRAIL] BLOCKED - "
            f"Semantic match: {matched_category}"
        )

        return (
            False,
            f"Input appears related to disallowed "
            f"content ({matched_category}). "
            f"This agent cannot assist with such requests."
        )


    # -----------------------------------------------------------
    # Otherwise allow
    # -----------------------------------------------------------

    print("[GUARDRAIL] PASSED - Input appears safe.")

    return True, "OK"

In [ ]:
# ---------------------------------------------------------------
# LLM GUARDRAIL - Semantic Input Validation
# ---------------------------------------------------------------

from typing import Tuple
from openai import OpenAI
import json


client = OpenAI()


GUARDRAIL_SYSTEM_PROMPT = """
You are a security and relevance guardrail for an AI task-processing agent.

Your job is to classify the user's request BEFORE it reaches the agent.

Classify the request into exactly one of:

1. SAFE
   - Normal, legitimate, benign requests.
   - General questions, research, writing, analysis, coding,
     business tasks, productivity, etc.

2. UNSAFE
   - Requests meaningfully intended to facilitate harmful,
     illegal, malicious, or dangerous activity.
   - This includes requests involving weapon construction,
     malicious cyber attacks, malware creation, credential theft,
     serious wrongdoing, or explicit sexual content.

3. OFF_TOPIC
   - Requests unrelated to the purpose of this task agent.

IMPORTANT:
- Judge the MEANING and INTENT of the request, not individual keywords.
- Do not block a request merely because it mentions a sensitive topic.
- Educational, defensive, historical, or safety-oriented questions
  can be SAFE when they do not meaningfully enable harmful activity.
- Be conservative about blocking legitimate questions.
- Do not answer the user's request. Only classify it.

Return ONLY valid JSON:

{
    "decision": "SAFE | UNSAFE | OFF_TOPIC",
    "reason": "short explanation",
    "confidence": 0.0
}
"""


def llm_guardrail_check(
    user_input: str
) -> Tuple[bool, str]:

    print(
        f"[GUARDRAIL] Checking input: "
        f"'{user_input[:80]}'"
    )

    # -----------------------------------------------------------
    # Basic validation
    # -----------------------------------------------------------

    if len(user_input.strip()) < 3:

        print("[GUARDRAIL] BLOCKED - Input too short.")

        return (
            False,
            "Input is too short. Please provide a meaningful question."
        )

    # -----------------------------------------------------------
    # Ask LLM to classify intent
    # -----------------------------------------------------------

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        response_format={
            "type": "json_object"
        },
        messages=[
            {
                "role": "system",
                "content": GUARDRAIL_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )

    # -----------------------------------------------------------
    # Parse response
    # -----------------------------------------------------------

    result = json.loads(
        response.choices[0].message.content
    )

    decision = result.get("decision", "UNSAFE")
    reason = result.get(
        "reason",
        "Unable to determine request safety."
    )
    confidence = result.get(
        "confidence",
        0.0
    )

    print(
        f"[GUARDRAIL] Decision: {decision} "
        f"| Confidence: {confidence}"
    )

    # -----------------------------------------------------------
    # Decision
    # -----------------------------------------------------------

    if decision == "SAFE":

        print("[GUARDRAIL] PASSED")

        return True, "OK"

    else:

        print(
            f"[GUARDRAIL] BLOCKED - "
            f"{decision}: {reason}"
        )

        return False, reason